<a href="https://colab.research.google.com/github/asoknaren/AIPlayground/blob/main/001_tokenizer_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

In [5]:
MODEL_NAME = "microsoft/phi-3-mini-4k-instruct"
TARGET_WORDS = 200

In [6]:
def build_prompt() -> str:
    return (
        "Write a sincere professional apology email in about 200 words. "
        "Context: I missed an important client deadline due to poor planning. "
        "The email should acknowledge responsibility, explain briefly without excuses, "
        "and include a clear corrective action plan and request to rebuild trust."
    )


In [7]:
prompt = build_prompt()

In [8]:
def initialize_model_and_tokenizer(model_name: str):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)

    

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    if device == "cpu":
        cpu_threads = max(1, (torch.get_num_threads() or 1) - 1)
        torch.set_num_threads(cpu_threads)
        print(f"Using CPU threads: {cpu_threads}")

    return tokenizer, model, device

In [9]:
tokenizer, model, device = initialize_model_and_tokenizer(MODEL_NAME)

print(model)  # Print the model details

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Using CPU threads: 1
Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_feat

## Show Tokens Helper Function

To better visualize the tokenization process, here's a helper function that displays tokens with different background colors.

In [8]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence):
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [9]:
show_tokens(prompt)

Write a s inc ere professional ap ology email in about  2 0 0 words . Context : I missed an important client dead line due to poor planning . The email should acknow ledge responsibility , explain briefly without exc uses , and include a clear correct ive action plan and request to re build trust . 

In [10]:
def tokenize_and_display_prompt_info(tokenizer, prompt: str, device: str):
    messages = [{"role": "user", "content": prompt}]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    token_count = model_inputs["input_ids"].shape[1]

    print(f"Prompt token count: {token_count}")
    print(f"Equivalent tokenized words: {tokenizer.decode(model_inputs['input_ids'][0])}")

    print("\nToken to Word Mapping:")
    for token_id in model_inputs['input_ids'][0]:
        token_word = tokenizer.decode(token_id)
        print(f"Token ID: {token_id.item()}, Word: '{token_word}'")

    return model_inputs, token_count

In [11]:
model_inputs, token_count = tokenize_and_display_prompt_info(tokenizer, prompt, device)

Prompt token count: 62
Equivalent tokenized words: <|user|> Write a sincere professional apology email in about 200 words. Context: I missed an important client deadline due to poor planning. The email should acknowledge responsibility, explain briefly without excuses, and include a clear corrective action plan and request to rebuild trust.<|end|><|assistant|>

Token to Word Mapping:
Token ID: 32010, Word: '<|user|>'
Token ID: 14350, Word: 'Write'
Token ID: 263, Word: 'a'
Token ID: 269, Word: 's'
Token ID: 3742, Word: 'inc'
Token ID: 406, Word: 'ere'
Token ID: 10257, Word: 'professional'
Token ID: 3095, Word: 'ap'
Token ID: 3002, Word: 'ology'
Token ID: 4876, Word: 'email'
Token ID: 297, Word: 'in'
Token ID: 1048, Word: 'about'
Token ID: 29871, Word: ''
Token ID: 29906, Word: '2'
Token ID: 29900, Word: '0'
Token ID: 29900, Word: '0'
Token ID: 3838, Word: 'words'
Token ID: 29889, Word: '.'
Token ID: 15228, Word: 'Context'
Token ID: 29901, Word: ':'
Token ID: 306, Word: 'I'
Token ID: 137

In [12]:
def generate_output(tokenizer, model, prompt: str, target_words: int, device: str) -> None:

    max_new_tokens = 240 if device == "cpu" else 320
    start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    elapsed_s = time.perf_counter() - start

    new_tokens = generated_ids[0, token_count:]
    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    generated_tokens = new_tokens.shape[0]
    tps = generated_tokens / elapsed_s if elapsed_s > 0 else 0.0
    print(
        f"Generated tokens: {generated_tokens} in {elapsed_s:.1f}s "
        f"({tps:.2f} tokens/sec, target words: {target_words})"
    )
    print("\nGenerated output:\n")
    print(output_text)




In [13]:
generate_output(tokenizer, model, prompt, TARGET_WORDS, device)

Generated tokens: 240 in 381.5s (0.63 tokens/sec, target words: 200)

Generated output:

Subject: Apology for Missed Deadline – Moving Forward Together


Dear [Client's Name],


I am writing to you with utmost sincerity regarding the recent delay of our project deliverable that was originally scheduled on [Date]. Please accept my heartfelt apologies as this oversight is not reflective of the high standards we set at [Your Company’s Name]. It has come to light through thorough review that there were shortcomings in our initial scheduling which led to today's unfortunate situation. While no one likes surprises or delays more than me personally, it would be remiss if I did not take full accountability for what happened here. My goal now—and always moving forward—is to ensure such instances are avoided entirely.


To rectify this matter promptly, we have implemented additional checks into all upcoming projects, including stricter timelines management protocols. This includes closer collabo